# vLLM vs Huggingface run time comparison

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/facebook/opt-125m

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/opt-125m)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
import time

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="facebook/opt-125m")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [3]:
user_input = "The capital of India is"

In [4]:
start_time = time.perf_counter()
results = pipe(user_input, do_sample=True, temperature=0.7)
end_time = time.perf_counter()
print(results)
execution_time = (end_time - start_time) * 1000
print(f"Execution time of Huggingface model: {execution_time:.3f} ms")

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'The capital of India is located in the heart of the ‘Golden Heart’, a prestigious honor bestowed annually for the outstanding contributions to sportsmanship and patriotism carried out by the nation. The Golden Heart recognizes individuals for their outstanding contributions to the national and international sporting scene and for their contributions to the development of the people of the country. The Golden Heart is also known as ‘Mansoora’, ‘Krishna’s Golden Heart’, and ‘Shirley’.\n\nThe Golden Heart is awarded on a merit scale. It is awarded for the outstanding achievements in sportsmanship, patriotism, and integrity. The Golden Heart recognizes individuals for their outstanding contributions to the national and international sporting scene and for their contributions to the development of the country. The Golden Heart is also known as ‘Mansoora’, ‘Krishna’s Golden Heart’, and ‘Shirley’.\n\nThe event is held in the presence of Prime Minister Narendra Modi, Presi

In [5]:
user_input = "The capital of India is"

In [6]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator

device = Accelerator().device

tokenizer = AutoTokenizer.from_pretrained("facebook/opt-125m")
model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m",dtype=torch.float16).to(device)


inputs = tokenizer(user_input, return_tensors="pt").to(device)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
start_time = time.perf_counter()
outputs = model.generate(**inputs) #max_new_tokens=20
results = tokenizer.batch_decode(outputs, skip_special_tokens=True)
end_time = time.perf_counter()
print(results)
execution_time = (end_time - start_time) * 1000
print(f"Execution time of Huggingface model: {execution_time:.3f} ms")

['The capital of India is the capital of the world.\n\nThe capital of the world is the capital of the world.']
Execution time of Huggingface model: 176.344 ms


## Inference with vLLM

In [ ]:
!uv pip install vllm --torch-backend=auto -q

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
# Sample prompts.
prompts = ["The capital of India is"]
# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.7) #, top_p=0.95

In [ ]:
llm = LLM(model="facebook/opt-125m")
start_time = time.perf_counter()
outputs = llm.generate(prompts, sampling_params)
end_time = time.perf_counter()

print("\nGenerated Outputs:\n" + "-" * 60)
print(outputs[0].outputs[0].text)

execution_time = (end_time - start_time) * 1000
print(f"Execution time of Huggingface model: {execution_time:.3f} ms")

INFO 05-10 12:36:54 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 05-10 12:36:54 [model.py:555] Resolved architecture: OPTForCausalLM
INFO 05-10 12:36:54 [model.py:1680] Using max model len 2048
INFO 05-10 12:36:54 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=20736) INFO 05-10 12:36:54 [core.py:109] Initializing a V1 LLM engine (v0.20.2) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_rou

(EngineCore pid=20736) Process EngineCore:
(EngineCore pid=20736) Traceback (most recent call last):
(EngineCore pid=20736)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=20736)     self.run()
(EngineCore pid=20736)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=20736)     self._target(*self._args, **self._kwargs)
(EngineCore pid=20736)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=20736)     raise e
(EngineCore pid=20736)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=20736)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=20736)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=20736)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wr

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
!python /content/sample_data/vLLM.py

INFO 05-10 12:40:41 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 05-10 12:40:41 [model.py:555] Resolved architecture: OPTForCausalLM
INFO 05-10 12:40:41 [model.py:1680] Using max model len 2048
INFO 05-10 12:40:41 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-10 12:40:41 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-10 12:40:41 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=22178) INFO 05-10 12:40:43 [core.py:109] Initializing a V1 LLM engine (v0.20.2) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, de